# 02: 原子发射物理 —— 780nm波包产生与SWAP传送带协议

## 概述

本notebook深入研究原子发射过程，验证**光子只在780nm子空间产生**，无泄漏到1517nm子空间。

### 物理过程

```
激发态原子 (|e>) → 780nm光子 → 基态原子 (|0> 或 |1>)
```

### 关键物理概念

1. **量子发射**：原子从激发态跃迁到基态，发射光子
2. **Time-bin波包**：光子分布在多个时间bin上，而非集中在一个模式
3. **SWAP传送带协议**：让原子沿链移动，确保每个bin只被耦合一次
4. **子空间隔离**：780nm和1517nm子空间完全独立（无QFC时）

---

## 1. Hilbert Space Definitions

### 1.1 Atomic System (3D)

$$
\mathcal{H}_{\mathrm{atom}} = \mathrm{span}\{|0\rangle, |1\rangle, |e\rangle\}
$$

- $|0\rangle, |1\rangle$: Two stable ground states (qubit)
- $|e\rangle$: Excited state for emission

### 1.2 Atomic Transition Operators

$$
S_{+} = |0\rangle\langle e|, \quad S_{-} = |1\rangle\langle e|
$$

**Selection rules**:
- $|e\rangle \to |0\rangle$: $\Delta m = +1$, emits sigma+ photon
- $|e\rangle \to |1\rangle$: $\Delta m = -1$, emits sigma- photon

### 1.3 Time-Bin Field Sites (18D)

$$
\mathcal{H}_{\mathrm{bin}} = \mathcal{H}_{780} \otimes \mathcal{H}_{1517}
$$

- **780nm subspace** (3D): $\{|\mathrm{vac}\rangle, |H\rangle, |V\rangle\}$
- **1517nm subspace** (6D): $\{|\mathrm{vac}\rangle, |H\rangle, |V\rangle, |2H\rangle, |2V\rangle, |HV\rangle\}$

**Index formula**:
$$
\mathrm{index} = i_{780} \times 6 + i_{1517}
$$

| $i_{780}$ | State | Index range |
|---------|-------|-------------|
| 0 | |vac⟩$_{780}$ | 0-5 |
| 1 | |H⟩$_{780}$ | 6-11 |
| 2 | |V⟩$_{780}$ | 12-17 |

### 1.4 MPS Chain Structure

$$
\mathrm{atom} - \mathrm{bin}_1 - \mathrm{bin}_2 - \cdots - \mathrm{bin}_N
$$

---

## 2. The Emission Gate $U^{(\mathrm{emit})}$

### 2.1 Coupling Operator

$$
L_{p}(t) = \sqrt{\gamma(t)} \left( \alpha_{p,+} S_{+} + \alpha_{p,-} S_{-} \right), \quad p \in \{H, V\}
$$

**Variables**:
- $\gamma(t)$: Emission rate (ns$^{-1}$)
- $\alpha_{p,\mu}$: Polarization mapping matrix element
- $S_{\pm}$: Atomic transition operators

### 2.2 Polarization Mapping Matrix $\alpha^{(\mathrm{emit})}$

$$
\alpha^{(\mathrm{emit})} = 
\begin{pmatrix}
\alpha_{H,+} & \alpha_{H,-} \\
\alpha_{V,+} & \alpha_{V,-}
\end{pmatrix}
$$

**Simple test config** (this notebook):
$$
\alpha = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}
$$

### 2.3 Emission Gate

$$
U^{(\mathrm{emit})} = \exp\left[ \sqrt{\Delta t} \sum_{p} \left( L_{p} b_{p}^\dagger - L_{p}^\dagger b_{p} \right) \right]
$$

**Variables**:
- $\Delta t$: Time step (ns), 0.2 ns in this notebook
- $b_{p}^\dagger$: 780nm photon creation operator
- $b_{p}$: 780nm photon annihilation operator

### 2.4 Embedding into Bin Space

$$
U_{54} = U_{9 \times 9} \otimes I_{1517}
$$

- $U_{9 \times 9}$: Acts on atom x 780(3D)
- $I_{1517}$: Identity on 1517(6D) subspace

**Key**: $I_{1517}$ ensures emission does NOT affect 1517nm subspace!

---

## 3. SWAP传送带协议 —— 正确的Time-Bin模型

### 3.1 为什么需要SWAP传送带？

**错误做法**（重复耦合同一个bin）：
- 每步都在(atom, site 1)上施加发射门
- 结果：原子与同一模式反复交换能量
- 现象：**拉比振荡**（能量来回跑），而非行波发射

**正确做法**（SWAP传送带）：
- 第n步让原子与第n个bin耦合
- 使用SWAP让原子沿链移动
- 结果：每个bin只被作用一次，光子分布在多个bins上

### 3.2 SWAP传送带算法

对 $n = 1, 2, \dots, N$:

1. **施加发射门**：在(atom, bin$_n$)上作用 $U^{(\mathrm{emit})}(\gamma_n)$
   $$
   |\Psi\rangle \leftarrow U^{(\mathrm{emit})}(\gamma_n)_{(\mathrm{atom}, \mathrm{bin}_n)} |\Psi\rangle
   $$

2. **记录概率**：该bin的占据概率
   $$
   p_n = \mathrm{Tr}[\rho_{\mathrm{bin}_n}, \Pi_{780}]
   $$

3. **SWAP前进**：交换原子与该bin的位置
   $$
   |\Psi\rangle \leftarrow W_{(\mathrm{atom}, \mathrm{bin}_n)} |\Psi\rangle
   $$

### 3.3 SWAP门定义

SWAP门 $W$ 交换两个量子态：
$$
W |s\rangle \otimes |t\rangle = |t\rangle \otimes |s\rangle
$$

对于原子(3D)与bin(18D)的交换，$W$是$54 \times 54$排列矩阵。

### 3.4 正确模型的预期结果

1. **非负柱状图**：每个bin的发射概率 $p_n \ge 0$
2. **单调递增累积**：$\sum_{k=1}^n p_k = 1 - P_e(n)$ 单调递增
3. **峰值量级**：$p_n \sim \gamma_n \Delta t \approx 0.04$（而非接近1）
4. **纠缠分布**：bond dimensions沿链分布（非只有第一条非1）

---

## 4. Gaussian Emission Rate Profile

$$
\gamma(t) = \gamma_{\mathrm{peak}} \exp\left(-\frac{(t - t_0)^2}{2\sigma^2}\right)
$$

**Variables**:
- $\gamma_{\mathrm{peak}}$: Peak emission rate
- $t_0$: Wavepacket center time
- $\sigma$: Gaussian width parameter
- FWHM $\approx 2.35\sigma$

---

## Part 1: 导入库与设置参数

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from atom_sim.simulation import run_dual_atom_emission
from atom_sim.visualization.wavepacket import extract_bin_state_probabilities

print("=" * 70)
print("Part 1: Imports and parameters")
print("=" * 70)

# Time grid
n_bins = 60
dt_ns = 0.2
chi_max = 50

# Emission profile
gamma_peak_A = 0.5
gamma_peak_B = 0.5
sigma = 10.0

delay_ns = 5.0
delay_jitter_ns = 0.0

seed = 123
rng = np.random.default_rng(seed)

print(f"N_bins = {n_bins}")
print(f"dt = {dt_ns} ns")
print(f"total time = {n_bins * dt_ns} ns")


---

## Part 2: 偏振映射矩阵 Alpha

**简单映射** (sigma+ -> H, sigma- -> V):

$$
\alpha = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix}
$$

In [ ]:
print("=" * 70)
print("Part 2: Polarization mapping")
print("=" * 70)

Alpha_A = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex)
Alpha_B = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex)

print("Alpha_A =")
print(Alpha_A)


---

## Part 3: 初始化MPS

**MPS链结构**:

$$
\mathrm{atom}(3D) - \mathrm{bin}_1(18D) - \mathrm{bin}_2(18D) - \cdots
$$

**初始态**:
- 原子：$|e\rangle$（激发态，索引2）
- 所有bins：$|\mathrm{vac}\rangle$（真空态，索引0）

In [ ]:
print("=" * 70)
print("Part 3: Run dual-atom emission")
print("=" * 70)

emission = run_dual_atom_emission(
    n_bins=n_bins,
    dt_ns=dt_ns,
    chi_max=chi_max,
    Alpha_A=Alpha_A,
    Alpha_B=Alpha_B,
    gamma_peak_A=gamma_peak_A,
    gamma_peak_B=gamma_peak_B,
    sigma=sigma,
    delay_ns=delay_ns,
    delay_jitter_ns=delay_jitter_ns,
    rng=rng,
    verbose=True,
)

mps = emission.mps
time_grid = emission.time_grid
time_ns = time_grid.t * 1e9

print(f"delay used = {emission.delay_ns_used:.2f} ns (jitter={emission.delay_jitter_actual_ns:.2f} ns)")
print(f"total emission prob: A={emission.per_bin_prob_A.sum():.6f}, B={emission.per_bin_prob_B.sum():.6f}")


---

## Part 4: SWAP传送带协议 —— Time-Bin发射

**算法**（对每个时间步 n = 0, 1, ..., N-1）:

1. 获取当前时间步的发射率 $\gamma_n$
2. 在(atom, 当前bin)上施加发射门 $U^{(\mathrm{emit})}(\gamma_n)$
3. 记录当前bin的780nm占据概率 $p_n$
4. SWAP原子与当前bin，原子移动到下一个位置

**关键**：每个bin只被耦合一次，不会有"再吸收"现象。

In [ ]:
print("=" * 70)
print("Part 4: 780nm wavepacket extraction")
print("=" * 70)

probs_A = extract_bin_state_probabilities(mps, arm='A', n_bins=n_bins)
probs_B = extract_bin_state_probabilities(mps, arm='B', n_bins=n_bins)

# 18D index: i_780 * 6 + j_1517
idx_780_H = 6
idx_780_V = 12

prob_780_H_A = probs_A[:, idx_780_H]
prob_780_V_A = probs_A[:, idx_780_V]
prob_780_total_A = prob_780_H_A + prob_780_V_A

prob_780_H_B = probs_B[:, idx_780_H]
prob_780_V_B = probs_B[:, idx_780_V]
prob_780_total_B = prob_780_H_B + prob_780_V_B

max_diff_A = float(np.max(np.abs(prob_780_total_A - emission.per_bin_prob_A)))
max_diff_B = float(np.max(np.abs(prob_780_total_B - emission.per_bin_prob_B)))

print(f"A arm total = {prob_780_total_A.sum():.6f}")
print(f"B arm total = {prob_780_total_B.sum():.6f}")
print(f"max diff vs emission summary: A={max_diff_A:.3e}, B={max_diff_B:.3e}")


---

## Part 5: 1517子空间泄漏检查

**理论预期**：由于没有QFC，1517子空间概率必须为零：

$$
P_{1517}^{(H)}(t) = P_{1517}^{(V)}(t) = 0
$$

In [ ]:
print("=" * 70)
print("Part 5: 1517nm leakage check")
print("=" * 70)

probs_A_reshaped = probs_A.reshape(n_bins, 3, 6)
probs_B_reshaped = probs_B.reshape(n_bins, 3, 6)

p_1517_A = probs_A_reshaped[:, :, 1:].sum(axis=(1, 2))
p_1517_B = probs_B_reshaped[:, :, 1:].sum(axis=(1, 2))

print(f"A arm 1517 sum = {p_1517_A.sum():.3e}, max/bin = {p_1517_A.max():.3e}")
print(f"B arm 1517 sum = {p_1517_B.sum():.3e}, max/bin = {p_1517_B.max():.3e}")


---

## Part 6: 波包分析

**每个bin的发射概率** $p_n$（这才是正确的"time-bin波包"表示）：

$$
p_n = \mathrm{Tr}[\rho_{\mathrm{bin}_n}, \Pi_{780}]
$$

**累积发射概率**：

$$
P_{\mathrm{cum}}(n) = \sum_{k=1}^n p_k = 1 - P_e(n)
$$

In [ ]:
print("=" * 70)
print("Part 6: Wavepacket analysis")
print("=" * 70)

cumulative_A = np.cumsum(prob_780_total_A)
cumulative_B = np.cumsum(prob_780_total_B)

peak_idx_A = int(np.argmax(prob_780_total_A))
peak_idx_B = int(np.argmax(prob_780_total_B))

print("A arm:")
print(f"  peak bin = {peak_idx_A}, t={time_ns[peak_idx_A]:.2f} ns")
print(f"  peak prob = {prob_780_total_A[peak_idx_A]:.6f}")
print(f"  total prob = {cumulative_A[-1]:.6f}")

print("B arm:")
print(f"  peak bin = {peak_idx_B}, t={time_ns[peak_idx_B]:.2f} ns")
print(f"  peak prob = {prob_780_total_B[peak_idx_B]:.6f}")
print(f"  total prob = {cumulative_B[-1]:.6f}")


---

## Part 7: 可视化

In [ ]:
print("=" * 70)
print("Part 7: Visualization")
print("=" * 70)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

ax = axes[0]
ax.plot(time_ns, prob_780_total_A, label='Total', color='tab:blue')
ax.plot(time_ns, prob_780_H_A, '--', label='H', color='tab:blue', alpha=0.6)
ax.plot(time_ns, prob_780_V_A, '--', label='V', color='tab:blue', alpha=0.9)
ax.set_title('Arm A (780nm)')
ax.set_xlabel('Time (ns)')
ax.set_ylabel('Probability')
ax.grid(True, alpha=0.3)
ax.legend()

ax = axes[1]
ax.plot(time_ns, prob_780_total_B, label='Total', color='tab:orange')
ax.plot(time_ns, prob_780_H_B, '--', label='H', color='tab:orange', alpha=0.6)
ax.plot(time_ns, prob_780_V_B, '--', label='V', color='tab:orange', alpha=0.9)
ax.set_title('Arm B (780nm)')
ax.set_xlabel('Time (ns)')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


---

## Part 8: 原子最终态分析

**理论预期**：经过完整演化后，原子应该主要在基态|0>和|1>。

注意：经过SWAP传送带后，原子从site 0移到了site N-1。

In [ ]:
print("=" * 70)
print("Part 8: Final atomic state")
print("=" * 70)

rho_A_final = emission.atom_states['A']
rho_B_final = emission.atom_states['B']

print("Atom A:")
print(f"  P(|0>) = {rho_A_final[0, 0].real:.6f}")
print(f"  P(|1>) = {rho_A_final[1, 1].real:.6f}")
print(f"  P(|e>) = {rho_A_final[2, 2].real:.6f}")

print("Atom B:")
print(f"  P(|0>) = {rho_B_final[0, 0].real:.6f}")
print(f"  P(|1>) = {rho_B_final[1, 1].real:.6f}")
print(f"  P(|e>) = {rho_B_final[2, 2].real:.6f}")


---

## Part 9: 物理解释

### 为什么SWAP传送带是正确的？

**行波发射的本质**：每个时间步耦合的是一个"新鲜真空模"，耦合完就再也不回来。这正是SWAP传送带模拟的行为。

**关键区别**：

| 特征 | 错误做法（重复耦合） | 正确做法（SWAP传送带） |
|------|---------------------|----------------------|
| 耦合对象 | 始终是bin1 | bin1, bin2, ..., binN |
| 物理图像 | 原子-单模交换 | 行波发射 |
| per-bin概率 | 可正可负 | 始终非负 |
| 累积概率 | 震荡 | 单调递增 |
| 峰值量级 | ~1（Rabi振荡） | ~0.04（γΔt） |

### 波包形状 vs γ(t)形状

在Markov发射中，真正的发射强度是：

$$
I(t) = \gamma(t) P_e(t), \quad \frac{dP_e}{dt} = -\gamma(t) P_e(t)
$$

因此：
$$
P_e(t) = \exp\left(-\int_0^t \gamma(s) ds\right), \quad
I(t) = \gamma(t) \exp\left(-\int_0^t \gamma(s) ds\right)
$$

波包形状 $I(t)$ 一般**不会**与 $\gamma(t)$ 完全相同，除非 $\gamma(t)$ 很弱。

In [ ]:
print("=" * 70)
print("Summary")
print("=" * 70)

print("1) 780nm emission only (1517 occupancy ~ 0)")
print("2) Conveyor-belt emission handled inside run_dual_atom_emission")
print("3) Wavepacket shape follows Gaussian gamma(t)")
